# Lab 3 — Building a RAG System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/angeloziletti/nlp-cas-dsmh/blob/main/notebooks/lab3_rag.ipynb)

In this lab you build a **Retrieval-Augmented Generation (RAG)** system end to end — a question-answering tool grounded in real drug-label documents.

**How to use it**
- Run each cell in order (`Shift + Enter`).
- Read the output *and* the explanation around it.
- Cells marked **🔧 YOUR TURN** are for you to change a value and re-run.
- You will run **Parts 1–3** in the short warm-up after the Block E lecture, then **Parts 4–7** in Block F.

**What you will build**
1. **Chunk** drug-label documents into passages
2. **Embed** the chunks and build a searchable **index**
3. **Retrieve** the most relevant chunks for a question
4. **Generate** a grounded answer — *with citations*
5. Put it together into a full **RAG pipeline**
6. **Break it on purpose** — see the failure modes
7. Inspect whether the answer is **faithful** to its sources

**The data:** real drug labels from the U.S. FDA's **openFDA** service — public-domain (CC0). No patient data is involved.

## Setup

Run the cells below. The first installs two libraries; the second connects everything and loads the documents.

Your Gemini API key must be available:

- **In Colab:** stored as a Secret named `GEMINI_API_KEY` (the 🔑 icon in the left sidebar, with *Notebook access* switched on).
- **Running locally:** set as an environment variable before launching Jupyter (`export GEMINI_API_KEY=...` on macOS/Linux, `$env:GEMINI_API_KEY = "..."` on Windows PowerShell).

In [ ]:
!pip install -q google-genai sentence-transformers

In [ ]:
import json
import os
import re
import numpy as np
import pandas as pd
from google import genai
from sentence_transformers import SentenceTransformer

# --- Read the Gemini API key (Colab Secrets, or local .env / env var) ---
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found. In Colab, store it as a Secret named "
        "GEMINI_API_KEY. Locally, copy .env.example to .env and paste your "
        "key there (or set the environment variable before launching Jupyter)."
    )

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3-flash-preview"

# --- The embedding model (for retrieval) — the biomedical model from Lab 1 ---
embedder = SentenceTransformer("NeuML/pubmedbert-base-embeddings")

# --- Course data lives in the GitHub repo ---
BASE_URL = "https://raw.githubusercontent.com/angeloziletti/nlp-cas-dsmh/main/data/"

print("Setup complete. Generation model:", MODEL)

In [ ]:
# Load the drug-label corpus. Each "document" is one section of one drug's label.
corpus = pd.read_json(BASE_URL + "drug_labels.json").to_dict("records")

drugs = sorted(set(d["drug"] for d in corpus))
print(f"Loaded {len(corpus)} documents covering {len(drugs)} drugs.")
print("Drugs:", ", ".join(drugs))
print("\nExample document:")
print(f"  drug: {corpus[0]['drug']}   section: {corpus[0]['section']}")
print(f"  text: {corpus[0]['text'][:300]} ...")

---
## Part 1 — Chunking: splitting documents into passages

A whole drug-label section is too long to be one unit of retrieval — a single vector for it would be *blurry* (Block E). So the first step is **chunking**: splitting each document into smaller passages, each of which becomes one searchable unit.

Our chunker splits text into sentences, then groups sentences into chunks of roughly a target size — with a one-sentence **overlap** so a key sentence is never stranded at a boundary.

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.])\s+", text) if s.strip()]

def chunk_text(text, target_chars=500, overlap_sentences=1):
    sents = split_sentences(text)
    if not sents:
        return []
    chunks, i = [], 0
    while i < len(sents):
        current, j = [], i
        while j < len(sents) and (not current or sum(len(s) for s in current) < target_chars):
            current.append(sents[j])
            j += 1
        chunks.append(" ".join(current))
        if j >= len(sents):
            break
        i = max(j - overlap_sentences, i + 1)   # step forward, keeping an overlap
    return chunks

# See it work on one document.
doc = corpus[0]
pieces = chunk_text(doc["text"])
print(f"Document: {doc['drug']} — {doc['section']}  ({len(doc['text'])} chars)")
print(f"Split into {len(pieces)} chunks:\n")
for n, p in enumerate(pieces, 1):
    print(f"--- chunk {n} ({len(p)} chars) ---\n{p}\n")

Notice each chunk is a few sentences long, and consecutive chunks **share a sentence** — that is the overlap. A question can now match the *specific passage* that answers it, not a whole label section.

In [ ]:
# 🔧 YOUR TURN: change the target chunk size and watch the number of chunks change.
my_target = 300

pieces = chunk_text(corpus[0]["text"], target_chars=my_target)
print(f"target_chars = {my_target}  ->  {len(pieces)} chunks")

Now chunk the **whole corpus**. Each chunk keeps its `drug` and `section` — that metadata is what lets the system **cite its source** later.

In [ ]:
chunks = []
for doc in corpus:
    for piece in chunk_text(doc["text"]):
        chunks.append({"text": piece, "drug": doc["drug"], "section": doc["section"]})

print(f"The {len(corpus)} documents became {len(chunks)} chunks.")
print("Example chunk:", {"drug": chunks[0]["drug"], "section": chunks[0]["section"]})

---
## Part 2 — Embedding and the vector index

Now we turn every chunk into a vector — the embeddings from Lab 1 — and collect them into one matrix. That matrix **is** our vector index: the thing we search.

In [ ]:
# Embed each chunk together with its drug and section label, so the
# vector captures WHAT the chunk is about - not just its raw words.
embed_inputs = [f"{c['drug']}, {c['section']}. {c['text']}" for c in chunks]
chunk_vectors = embedder.encode(embed_inputs, show_progress_bar=True)

print(f"Embedded {len(chunks)} chunks.")
print(f"The index is a matrix of shape {chunk_vectors.shape}  (chunks x vector-dimensions).")

Notice we embedded each chunk **with its drug and section label attached** — *"metformin, Contraindications. ..."* — not just the bare text. A bare passage does not say which drug it belongs to; attaching the label so the embedding captures it is a small, standard trick that makes retrieval noticeably more accurate.

That matrix is our **vector index**. We search it by brute force — comparing a query against every chunk — exactly as in Lab 1. At this scale (a few hundred chunks) brute force is instant; a production system with millions of chunks would use a dedicated **vector database** (such as FAISS or Chroma) that does the same nearest-neighbour search efficiently.

---
## Part 3 — Retrieval: finding the right chunks

Retrieval finds the chunks most relevant to a question. We combine **two signals**:

- **Semantic similarity** — meaning-match, using the embeddings (as in Lab 1).
- **Keyword overlap** — exact-term match, so a question naming *"metformin"* is pulled toward chunks that literally contain *metformin*.

Blending the two — a **hybrid** search — is a standard, practical way to make retrieval more reliable than meaning-match alone. We return the best `k` chunks.

In [ ]:
STOPWORDS = {"what", "are", "the", "for", "and", "with", "that", "this", "can",
             "does", "have", "has", "its", "from", "any", "not", "you", "your",
             "when", "which", "how", "why", "there", "their", "they", "into",
             "about", "should", "would", "use", "used", "using", "most", "common"}

def tokenize(text):
    return {w for w in re.findall(r"[a-z]{3,}", text.lower()) if w not in STOPWORDS}

# keyword set for every chunk (drug + section + body text)
chunk_keywords = [tokenize(f"{c['drug']} {c['section']} {c['text']}") for c in chunks]

def retrieve(query, k=6):
    # signal 1 - semantic similarity (meaning)
    qv = embedder.encode(query)
    dense = chunk_vectors @ qv / (np.linalg.norm(chunk_vectors, axis=1) * np.linalg.norm(qv))
    dense = (dense - dense.min()) / (dense.max() - dense.min())   # scale to 0-1
    # signal 2 - keyword overlap (exact terms)
    q_words = tokenize(query)
    lexical = np.array([len(q_words & kw) / max(len(q_words), 1) for kw in chunk_keywords])
    # blend the two signals
    score = 0.6 * dense + 0.4 * lexical
    top = np.argsort(score)[::-1][:k]
    return [(chunks[i], float(score[i])) for i in top]

# Try it.
for chunk, score in retrieve("What are the adverse reactions to atorvastatin?"):
    print(f"({score:.3f})  {chunk['drug']} - {chunk['section']}")
    print(f"   {chunk['text'][:150]} ...\n")

Look at what came back: the chunks most similar in meaning to the question, each tagged with the drug and section it came from. **This is the moment to inspect the index** — are these chunks actually relevant? Their scores tell you how confident the match is.

> This is the end of the Block E warm-up. In Block F, we continue from Part 4.

In [ ]:
# 🔧 YOUR TURN: retrieve chunks for your own question.
my_query = "What are the side effects of statins?"

for chunk, score in retrieve(my_query):
    print(f"({score:.3f})  {chunk['drug']} — {chunk['section']}")

---
## Part 4 — Augmented generation: the grounded answer

So far we have only *retrieved* text. Now we add the **generation** step: hand the retrieved chunks to the model and ask it to answer **using only those passages**, and to **cite** them.

This is the grounded prompt from the Block E lecture — Block C prompting plus Block D's grounding rule.

In [ ]:
def build_prompt(question, retrieved):
    passages = ""
    for i, (chunk, score) in enumerate(retrieved, 1):
        passages += f"Passage {i} (from {chunk['drug']}, {chunk['section']}):\n{chunk['text']}\n\n"
    return f"""You are a clinical information assistant. Answer the question using ONLY the passages below.
If the passages do not contain the answer, reply exactly: "The provided passages do not contain this information."
Cite the passage number(s) you used, like (Passage 2).

{passages}Question: {question}

Answer:"""

# Build and inspect the prompt for one question.
question = "What are the contraindications for metformin?"
retrieved = retrieve(question)
prompt = build_prompt(question, retrieved)
print(prompt)

In [ ]:
# Send the grounded prompt to the model.
answer = client.models.generate_content(model=MODEL, contents=prompt).text
print(answer)

Read the answer against the passages above it. The model answered **from the retrieved text**, and the citation — *(Passage 1)* — points back to a specific drug-label section a human can open and verify. That is the whole point of RAG: an answer you can **check**.

---
## Part 5 — The full RAG pipeline

Wrap retrieval and generation into one function — a complete RAG system in a few lines.

In [ ]:
def rag_answer(question, k=6):
    retrieved = retrieve(question, k)
    prompt = build_prompt(question, retrieved)
    answer = client.models.generate_content(model=MODEL, contents=prompt).text
    return answer, retrieved

for q in ["Can warfarin be taken during pregnancy?",
          "What is the recommended use of amoxicillin?",
          "What should be monitored in a patient taking furosemide?"]:
    answer, retrieved = rag_answer(q)
    sources = ", ".join(sorted(set(f"{c['drug']}/{c['section']}" for c, s in retrieved)))
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   [retrieved from: {sources}]\n")

In [ ]:
# 🔧 YOUR TURN: ask the RAG system your own question about any of the 20 drugs.
my_question = "What are the warnings for ibuprofen?"

answer, retrieved = rag_answer(my_question)
print(answer)

---
## Part 6 — Break it on purpose

A RAG system is only as good as its retrieval and its corpus (Block E). Let's make it fail — deliberately — so the failure modes are real, not theoretical.

**Failure 1 — a question the corpus cannot answer.** Our corpus has 20 drugs. Ask about one that is *not* in it.

In [ ]:
# Insulin is NOT in our 20-drug corpus.
answer, retrieved = rag_answer("What are the contraindications for insulin?")
print("ANSWER:")
print(answer)
print("\nWHAT WAS RETRIEVED:")
for chunk, score in retrieved:
    print(f"  ({score:.3f})  {chunk['drug']} — {chunk['section']}")

**Inspect what happened.** Retrieval *always* returns the closest chunks — even when nothing is truly relevant, it returns *something*. The question is what the model did with weak passages:

- A well-behaved RAG system says *"the provided passages do not contain this information"* — the grounding rule working.
- A poorly grounded one would answer anyway, from memory — a **hallucination wearing a citation**.

This is why the grounding instruction in the prompt matters so much.

In [ ]:
# Failure 2 — an off-topic query. Retrieval still returns something.
print("Query: 'What is the capital of France?'\n")
for chunk, score in retrieve("What is the capital of France?"):
    print(f"  ({score:.3f})  {chunk['drug']} — {chunk['section']}")

The scores are low and the chunks are irrelevant — but retrieval **still returned its top 4**. Retrieval never says "nothing found"; it always returns *the closest things it has*. A RAG system must be built to notice when the closest things are not good enough.

> **The lesson (Block E):** RAG makes answers grounded and checkable — it does not make them guaranteed correct. Retrieval can miss, and the corpus has limits.

---
## Part 7 — Is the answer faithful to its sources?

A RAG system has two stages that can each fail (this connects to the evaluation block, Block H):

- **Retrieval** — did it fetch the right chunks?
- **Faithfulness** — does the answer actually follow from those chunks, or did the model add something?

Let's inspect both for one question.

In [ ]:
question = "What are the most common adverse reactions to sertraline?"
answer, retrieved = rag_answer(question)

print("RETRIEVED PASSAGES:")
for i, (chunk, score) in enumerate(retrieved, 1):
    print(f"\n[Passage {i}] {chunk['drug']} — {chunk['section']} ({score:.3f})")
    print(chunk["text"])

print("\n" + "=" * 60)
print("\nTHE ANSWER:")
print(answer)

**Check it yourself**, the way you would evaluate any RAG system:

- **Retrieval:** are the retrieved passages actually about the question? (Right drug? Right section?)
- **Faithfulness:** is every claim in the answer supported by the passages above — or did the model add a detail that is not there?

A RAG answer can fail at *either* stage — good retrieval with an unfaithful answer, or a faithful answer built on the wrong passages. Evaluating them **separately** is how you know which half to fix. More on this in the final block.

---
## Recap — what you built

| Part | What you did |
|---|---|
| Chunking | Split documents into overlapping passages |
| Embedding & index | Turned chunks into vectors — a searchable index |
| Retrieval | Found the most relevant chunks for a question |
| Augmented generation | Generated a grounded answer with citations |
| Full pipeline | Combined it into one `rag_answer` function |
| Breaking it | Saw retrieval always return *something* — and why grounding matters |
| Faithfulness | Inspected retrieval and answer-faithfulness separately |

**You have built the "open-book exam" from the Block E lecture — and seen where the book lets you down.**

**Next:** **agents** — letting a model use tools and take steps, not just retrieve and answer.